<a href="https://colab.research.google.com/github/abdullahamjad20/Artifical-Intelligence-Lab/blob/main/AI%20Lab%201/task2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
import random

MOVE_PENALTY = 1
STEPS = 20


def random_initial_state(rng):
    return rng.choice("AB"), rng.random() < 0.5, rng.random() < 0.5


def step(state, action):
    loc, dirt_a, dirt_b = state
    dirt = {"A": dirt_a, "B": dirt_b}
    penalty = 0
    if action == "Suck":
        dirt[loc] = False
    elif action == "Left":
        loc, penalty = "A", MOVE_PENALTY
    elif action == "Right":
        loc, penalty = "B", MOVE_PENALTY
    # NoOp: nothing happens, no penalty
    reward = (not dirt["A"]) + (not dirt["B"]) - penalty
    return (loc, dirt["A"], dirt["B"]), reward


class SimpleReflexAgent:
    def act(self, percept):
        loc, dirty = percept
        if dirty:
            return "Suck"
        return "Right" if loc == "A" else "Left"


class ModelBasedReflexAgent:
    """Remembers which squares are known to be clean."""
    def __init__(self):
        self.clean = {"A": False, "B": False}

    def act(self, percept):
        loc, dirty = percept
        if not dirty:
            self.clean[loc] = True
        if self.clean["A"] and self.clean["B"]:
            return "NoOp"
        if dirty:
            self.clean[loc] = True      # will be clean after sucking
            return "Suck"
        return "Right" if loc == "A" else "Left"


def run_episode(agent, state, steps=STEPS):
    total = 0
    for _ in range(steps):
        loc, dirt_a, dirt_b = state
        percept = (loc, dirt_a if loc == "A" else dirt_b)
        state, reward = step(state, agent.act(percept))
        total += reward
    return total


seed, n_states = 7, 10
rng = random.Random(seed)
states = [random_initial_state(rng) for _ in range(n_states)]

print(f"Move penalty = {MOVE_PENALTY}, steps = {STEPS}\n")
print(f"{'Run':<4}{'Initial state':<40}{'Reflex':<8}{'Model':<8}")
rs, ms = [], []
for i, s in enumerate(states, 1):
    a = run_episode(SimpleReflexAgent(), s)
    b = run_episode(ModelBasedReflexAgent(), s)
    rs.append(a)
    ms.append(b)
    desc = f"loc={s[0]}, A_dirty={s[1]}, B_dirty={s[2]}"
    print(f"{i:<4}{desc:<40}{a:<8}{b:<8}")

avg_r, avg_m = sum(rs) / n_states, sum(ms) / n_states
print(f"\nAverage Simple Reflex score : {avg_r:.2f}")
print(f"Average Model-based score   : {avg_m:.2f}")
print(f"Model-based agent higher?   : {avg_m > avg_r}")

# Exhaustive check over all 8 possible initial states
print("\nAll 8 possible initial states:")
all_ok = True
for loc in "AB":
    for da in (True, False):
        for db in (True, False):
            s = (loc, da, db)
            a = run_episode(SimpleReflexAgent(), s)
            b = run_episode(ModelBasedReflexAgent(), s)
            all_ok &= (b >= a)
            print(f"loc={loc}, A_dirty={da!s:<5}, B_dirty={db!s:<5} -> reflex={a}, model={b}")
print(f"\nModel-based never worse than reflex: {all_ok}")
print("Reason: after both squares are clean the reflex agent keeps moving and pays "
      "the penalty,\nwhile the model-based agent issues NoOp.")

Move penalty = 1, steps = 20

Run Initial state                           Reflex  Model   
1   loc=B, A_dirty=False, B_dirty=True      21      39      
2   loc=A, A_dirty=True, B_dirty=False      21      39      
3   loc=B, A_dirty=False, B_dirty=False     20      39      
4   loc=A, A_dirty=True, B_dirty=True       20      37      
5   loc=A, A_dirty=True, B_dirty=False      21      39      
6   loc=A, A_dirty=False, B_dirty=True      20      38      
7   loc=A, A_dirty=False, B_dirty=False     20      39      
8   loc=A, A_dirty=False, B_dirty=True      20      38      
9   loc=A, A_dirty=True, B_dirty=False      21      39      
10  loc=B, A_dirty=True, B_dirty=False      20      38      

Average Simple Reflex score : 20.40
Average Model-based score   : 38.50
Model-based agent higher?   : True

All 8 possible initial states:
loc=A, A_dirty=True , B_dirty=True  -> reflex=20, model=37
loc=A, A_dirty=True , B_dirty=False -> reflex=21, model=39
loc=A, A_dirty=False, B_dirty=True  -> re